# Resume Data Exploration & ML Model Development

This notebook explores the large resume dataset (`resume_data.csv`) and develops improved ML models for the AI-Powered Multimodal Smart Recruitment Platform.

**Dataset Overview:**
- File: resume_data.csv
- Size: ~17MB
- Rows: 217,473
- Columns: 35 (plus some corrupted data)

**Goal:**
1. Clean and understand the dataset structure
2. Extract meaningful features for resume parsing and job matching
3. Train improved classification and matching models
4. Evaluate model performance
5. Provide recommendations for production use

In [0]:
# Import necessary libraries
import pandas as pd
import numpy as np
import json
import re
import ast
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# ML libraries
from sklearn.model import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

In [1]:
# Load the dataset
print("Loading resume_data.csv...")
df = pd.read_csv('/d/files/resume_data.csv', nrows=1000)  # Start with sample
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
# Show first few rows
display(df.head(3))

Loading resume_data.csv...
Dataset shape: (1000, 35)
Columns: ['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', '\ufeffjob_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']


In [2]:
# Check data types and missing values
print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES (Top 10) ===")
missing = df.isnull().sum().sort_values(ascending=False)
display(missing.head(10))

print("\n=== MEMORY USAGE ===")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


=== DATA TYPES ===
address                                 object
career_objective                        object
skills                                  object
educational_institution_name            object
degree_names                            object
passing_years                           object
educational_results                     object
result_types                            object
major_field_of_studies                  object
professional_company_names              object
company_urls                            object
start_dates                             object
end_dates                               object
related_skils_in_job                    object
positions                               object
locations                               object
responsibilities                        object
extra_curricular_activity_types         object
extra_curricular_organization_names     object
extra_curricular_organization_links     object
role_positions                          

In [3]:
# Clean column names (remove BOM and fix typos)
df_clean = df.copy()
df_clean.columns = [
    col.replace('����������������������������������������������', '').replace('educationaL', 'education').replace('experiencere', 'experience').replace('responsibilities.1', 'job_responsibilities')
    for col in df_clean.columns
]

print("Cleaned column names:")
for i, col in enumerate(df_clean.columns):
    print(f"{i+1:2d}. {col}")

In [4]:
# Helper function to safely parse array-like strings
def parse_array_string(val):
    """Safely parse string that looks like a Python list"""
    if pd.isna(val) or val == '' or val == '[]':
        return []
    try:
        # Remove extra whitespace
        val = val.strip()
        # If it looks like a Python list
        if val.startswith('[') and val.endswith(']'):
            return ast.literal_eval(val)
        # If it's a comma-separated string without brackets
        elif ',' in val:
            return [item.strip().strip("'\"") for item in val.split(',') if item.strip()]
        # Single item
        else:
            return [val.strip().strip("'\"")] if val.strip() else []
    except (ValueError, SyntaxError):
        # Fallback: split by commars and clean
        return [item.strip().strip("'\"") for item in val.split(',') if item.strip()]

# Test the parser
test_samples = [
    "['Big Data', 'Hadoop', 'Python']",
    "Big Data, Hadoop, Python",
    "Machine Learning",
    "",
    None
]

print("Testing array parser:")
for sample in test_samples:
    print(f"  {repr(sample)} -> {parse_array_string(sample)}")

In [5]:
# Feature Engineering: Create rich resume and job description texts
print("Engineering features...")

# 1. Resume Text (combines candidate profile information)
def create_resume_text(row):
    parts = []

    # Career objective/summary
    if pd.notna(row['career_objective']) and row['career_objective'].strip():
        parts.append(str(row['career_objective']).strip())

    # Skills
    skills_list = parse_array_string(row['skills'])
    if skills_list:
        parts.append('Skills: ' + ', '.join(skills_list))

    # Education
    edu_parts = []
    if pd.notna(row['educational_institution_name']) and row['educational_institution_name'].strip():
        edu_inst = parse_array_string(row['educational_institution_name'])
        if edu_inst:
            edu_parts.append('Institution: ' + ', '.join(edu_inst))
    
    if pd.notna(row['degree_names']) and row['degree_names'].strip():
        degrees = parse_array_string(row['degree_names'])
        if degrees:
            edu_parts.append('Degrees: ' + ', '.join(degrees))
    
    if pd.notna(row['passing_years']) and row['passing_years'].strip():
        years = parse_array_string(row['passing_years'])
        if years:
            edu_parts.append('Years: ' + ', '.join(years))
    
    if edu_parts:
        parts.append('Education: ' + ' | '.join(edu_parts))

    # Experience
    exp_parts = []
    if pd.notna(row['professional_company_names']) and row['professional_company_names'].strip():
        companies = parse_array_string(row['professional_company_names'])
        if companies:
            exp_parts.append('Companies: ' + ', '.join(companies))
    
    if pd.notna(row['positions']) and row['positions'].strip():
        positions = parse_array_string(row['positions'])
        if positions:
            exp_parts.append('Positions: ' + ', '.join(positions))
    
    if pd.notna(row['responsibilities']) and row['responsibilities'].strip():
        resp = parse_array_string(row['responsibilities'])
        if resp:
            exp_parts.append('Responsibilities: ' + ', '.join(resp))
    
    if exp_parts:
        parts.append('Experience: ' + ' | '.join(exp_parts))

    # Certifications
    if pd.notna(row['certification_skills']) and row['certification_skills'].strip():
        certs = parse_array_string(row['certification_skills'])
        if certs:
            parts.append('Certifications: ' + ', '.join(certs))

    return ' \n '.join(parts)

# 2. Job Description Text
def create_job_text(row):
    parts = []

    # Job title
    if pd.notna(row['job_position_name']) and row['job_position_name'].strip():
        parts.append('Position: ' + str(row['job_position_name']).strip())

    # Job responsibilities
    if pd.notna(row['job_responsibilities']) and row['job_responsibilities'].strip():
        resp_list = parse_array_string(row['job_responsibilities'])
        if resp_list:
            parts.append('Responsibilities: ' + ', '.join(resp_list))

    # Required skills
    if pd.notna(row['skills_required']) and row['skills_required'].strip():
        skills_req = parse_array_string(row['skills_required'])
        if skills_req:
            parts.append('Required Skills: ' + ', '.join(skills_req))

    # Education requirements
    if pd.notna(row['education_requirements']) and row['education_requirements'].strip():
        edu_req = parse_array_string(row['education_requirements'])
        if edu_req:
            parts.append('Required Education: ' + ', '.join(edu_req))

    # Experience requirements
    if pd.notna(row['experience_requirement']) and row['experience_requirement'].strip():
        exp_req = parse_array_string(row['experience_requirement'])
        if exp_req:
            parts.append('Required Experience: ' + ', '.join(exp_req))

    # Age requirements
    if pd.notna(row['age_requirement']) and str(row['age_requirement']).strip():
        parts.append('Age Requirement: ' + str(row['age_requirement']).strip())

    return ' \n '.join(parts)

# Apply feature engineering
df_clean['resume_text'] = df_clean.apply(create_resume_text, axis=1)
df_clean['job_text'] = df_clean.apply(create_job_text, axis=1)

# Clean matched_score for regression/classification
df_clean['matched_score_clean'] = pd.to_numeric(df_clean['matched_score'], errors='coerce')

print(f"Resume text sample:\n{df_clean['resume_text'].iloc[0][:200]}...")
print(f"\nJob text sample:\n{df_clean['job_text'].iloc[0][:200]}...")
print(f"\nMatched score stats:\n{df_clean['matched_score_clean'].describe()}")

In [6]:
# Exploratory Data Analysis
print("=== EXPLORATORY DATA ANALYSIS ===")

# 1. Matched score distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df_clean['matched_score_clean'].dropna(), bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of Match Scores')
plt.xlabel('Match Score')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(df_clean['matched_score_clean'].dropna())
plt.title('Match Score Box Plot')
plt.ylabel('Match Score')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 2. Text length analysis
df_clean['resume_length'] = df_clean['resume_text'].str.len()
df_clean['job_length'] = df_clean['job_text'].str.len()

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df_clean['resume_length'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
plt.title('Resume Text Length Distribution')
plt.xlabel('Text Length (characters)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(df_clean['job_length'], bins=30, edgecolor='black', alpha=0.7, color='lightcoral')
plt.title('Job Text Length Distribution')
plt.xlabel('Text Length (characters)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Resume length stats: mean={df_clean['resume_length'].mean():.0f}, median={df_clean['resume_length'].median():.0f}")
print(f"Job length stats: mean={df_clean['job_length'].mean():.0f}, median={df_clean['job_length'].median():.0f}")